In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, silhouette_score


print("=" * 80)
print("NASA TURBOFAN ENGINE ANOMALY DETECTION ANALYSIS")
print("Using DBSCAN and Isolation Forest for Engine Failure Prediction")
print("=" * 80)

print("\nSTEP 1: UNDERSTANDING SENSOR READINGS AND UNIT IDs")
print("-" * 60)

np.random.seed(42)
n_engines = 100
max_cycles_per_engine = 200

print("Creating synthetic NASA Turbofan dataset...")
print("(Replace this section with: df = pd.read_csv('NASA_Turbofan_Dataset.csv'))")

all_data = []
for engine in range(1, n_engines + 1):
    engine_lifetime = np.random.randint(50, max_cycles_per_engine)
    
    for cycle in range(1, engine_lifetime + 1):
        degradation_factor = cycle / engine_lifetime
        
        row = {
            'unit_id': engine,
            'cycle': cycle,
            'setting_1': np.random.normal(-0.0007, 0.0005),
            'setting_2': np.random.normal(0.0003, 0.0002),
            'setting_3': np.random.normal(100, 1),
            # Temperature sensors (°R) - increase with degradation
            'T2': np.random.normal(518.67, 2) + degradation_factor * np.random.normal(5, 1),
            'T24': np.random.normal(642.6, 5) + degradation_factor * np.random.normal(8, 2),
            'T30': np.random.normal(1589.7, 10) + degradation_factor * np.random.normal(15, 3),
            'T50': np.random.normal(1400.6, 15) + degradation_factor * np.random.normal(12, 4),
            # Pressure sensors (psia) - decrease with degradation
            'P2': np.random.normal(14.62, 0.1) + degradation_factor * np.random.normal(-0.2, 0.05),
            'P15': np.random.normal(21.6, 0.5) + degradation_factor * np.random.normal(-0.3, 0.1),
            'P30': np.random.normal(553.9, 10) + degradation_factor * np.random.normal(-5, 2),
            # Speed sensors (rpm) - decrease with degradation
            'Nf': np.random.normal(2388.1, 20) + degradation_factor * np.random.normal(-10, 5),
            'Nc': np.random.normal(9062.2, 100) + degradation_factor * np.random.normal(-50, 15),
            # Other sensors
            'epr': np.random.normal(1.3, 0.02) + degradation_factor * np.random.normal(-0.01, 0.005),
            'Ps30': np.random.normal(47.5, 1) + degradation_factor * np.random.normal(-0.5, 0.2),
            'phi': np.random.normal(0.84, 0.01) + degradation_factor * np.random.normal(0.02, 0.005),
            'NRf': np.random.normal(2388.1, 20) + degradation_factor * np.random.normal(-8, 4),
            'NRc': np.random.normal(9062.2, 100) + degradation_factor * np.random.normal(-45, 12),
            'BPR': np.random.normal(8.4, 0.1) + degradation_factor * np.random.normal(-0.05, 0.02),
            'farB': np.random.normal(0.03, 0.001) + degradation_factor * np.random.normal(0.005, 0.001),
            'htBleed': np.random.normal(391, 5) + degradation_factor * np.random.normal(3, 1),
            'RUL': engine_lifetime - cycle,  # Remaining Useful Life
            'is_anomaly': 1 if (engine_lifetime - cycle) <= 30 else 0  # Failure warning
        }
        all_data.append(row)

df = pd.DataFrame(all_data)
sensor_cols = ['T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 
               'epr', 'Ps30', 'phi', 'NRf', 'NRc', 'BPR', 'farB', 'htBleed']

print(f"✓ Dataset: {df.shape[0]} observations from {df['unit_id'].nunique()} engines")
print(f"✓ Sensors: {len(sensor_cols)} sensor measurements")
print(f"✓ Anomaly rate: {(df['is_anomaly'].sum() / len(df) * 100):.1f}%")

print("\nSTEP 2: CLEANING AND PREPROCESSING TIME-SERIES DATA")
print("-" * 60)

print("Data Quality Check:")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

outlier_count = 0
for sensor in sensor_cols:
    Q1 = df[sensor].quantile(0.25)
    Q3 = df[sensor].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[sensor] < lower_bound) | (df[sensor] > upper_bound)]
    outlier_count += len(outliers)

print(f"Total outlier data points detected: {outlier_count}")
print("✓ Data preprocessing completed")

print("\nSTEP 3: AGGREGATING DATA BY CYCLE AND STATISTICAL SUMMARIES")
print("-" * 60)

engine_stats = []
for engine_id in df['unit_id'].unique():
    engine_data = df[df['unit_id'] == engine_id]
    
    stats = {
        'unit_id': engine_id,
        'total_cycles': len(engine_data),
        'final_rul': engine_data['RUL'].iloc[-1]
    }
    
    for sensor in sensor_cols[:10]:  # Use subset for demonstration
        stats[f'{sensor}_mean'] = engine_data[sensor].mean()
        stats[f'{sensor}_std'] = engine_data[sensor].std()
        stats[f'{sensor}_max'] = engine_data[sensor].max()
        stats[f'{sensor}_min'] = engine_data[sensor].min()
        stats[f'{sensor}_trend'] = np.polyfit(engine_data['cycle'], engine_data[sensor], 1)[0]
    
    engine_stats.append(stats)

df_aggregated = pd.DataFrame(engine_stats)
print(f"✓ Aggregated dataset: {df_aggregated.shape[0]} engines with {df_aggregated.shape[1]-1} features")

print("\nSTEP 4: DIMENSIONALITY REDUCTION USING PCA")
print("-" * 60)

feature_cols = [col for col in df_aggregated.columns if col not in ['unit_id', 'total_cycles', 'final_rul']]
X = df_aggregated[feature_cols].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=0.95)  # Keep 95% of variance
X_pca = pca.fit_transform(X_scaled)

print(f"Original features: {X_scaled.shape[1]}")
print(f"PCA components: {X_pca.shape[1]}")
print(f"Variance explained: {pca.explained_variance_ratio_.sum():.3f}")
print(f"Top 5 components variance: {pca.explained_variance_ratio_[:5].round(3)}")

print("\nSTEP 5: APPLYING DBSCAN FOR ANOMALY DETECTION")
print("-" * 60)

best_eps = 2.0
best_min_samples = 5

dbscan = DBSCAN(eps=best_eps, min_samples=best_min_samples)
dbscan_labels = dbscan.fit_predict(X_pca)

# Calculate results
n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)
noise_ratio = n_noise / len(dbscan_labels)

df_aggregated['dbscan_cluster'] = dbscan_labels
df_aggregated['dbscan_anomaly'] = (dbscan_labels == -1).astype(int)

print(f"✓ DBSCAN Results:")
print(f"  Clusters found: {n_clusters}")
print(f"  Anomalies (noise): {n_noise} ({noise_ratio*100:.1f}%)")
print(f"  Parameters used: eps={best_eps}, min_samples={best_min_samples}")

print("\nSTEP 6: PREPARING 2D VISUALIZATION DATA")
print("-" * 60)

pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

df_aggregated['pca_1'] = X_pca_2d[:, 0]
df_aggregated['pca_2'] = X_pca_2d[:, 1]

print(f"✓ 2D PCA completed")
print(f"  PC1 variance explained: {pca_2d.explained_variance_ratio_[0]:.3f}")
print(f"  PC2 variance explained: {pca_2d.explained_variance_ratio_[1]:.3f}")

print("\nSTEP 7: APPLYING ISOLATION FOREST FOR COMPARISON")
print("-" * 60)

contamination_rate = 0.2
iso_forest = IsolationForest(contamination=contamination_rate, random_state=42, n_estimators=100)
iso_labels = iso_forest.fit_predict(X_pca)
iso_scores = iso_forest.decision_function(X_pca)

df_aggregated['iso_forest_label'] = iso_labels
df_aggregated['iso_forest_anomaly'] = (iso_labels == -1).astype(int)
df_aggregated['iso_forest_score'] = iso_scores

n_iso_anomalies = (iso_labels == -1).sum()

print(f"✓ Isolation Forest Results:")
print(f"  Anomalies detected: {n_iso_anomalies} ({n_iso_anomalies/len(iso_labels)*100:.1f}%)")
print(f"  Contamination rate: {contamination_rate}")
print(f"  Score range: {iso_scores.min():.3f} to {iso_scores.max():.3f}")

print("\nSTEP 8: IDENTIFYING EARLY WARNINGS OF ENGINE FAILURE")
print("-" * 60)

df_aggregated['failure_risk'] = (df_aggregated['final_rul'] <= 30).astype(int)

early_warnings = df_aggregated['failure_risk'].sum()
print(f"✓ Early Warning Analysis:")
print(f"  Engines at risk (RUL ≤ 30): {early_warnings}")
print(f"  Risk rate: {early_warnings/len(df_aggregated)*100:.1f}%")

risk_engines = df_aggregated[df_aggregated['failure_risk'] == 1]
healthy_engines = df_aggregated[df_aggregated['failure_risk'] == 0]

key_sensors = ['T30', 'T50', 'P30', 'Nf', 'Nc']
print(f"\nSensor Trend Comparison (Risk vs Healthy):")
for sensor in key_sensors:
    if f'{sensor}_trend' in df_aggregated.columns:
        risk_trend = risk_engines[f'{sensor}_trend'].mean()
        healthy_trend = healthy_engines[f'{sensor}_trend'].mean()
        print(f"  {sensor}: Risk={risk_trend:.4f}, Healthy={healthy_trend:.4f}")

print("\nSTEP 9: LABELING AND EVALUATING OUTLIER DETECTION")
print("-" * 60)

y_true = df_aggregated['failure_risk']
y_pred_dbscan = df_aggregated['dbscan_anomaly']
y_pred_iso = df_aggregated['iso_forest_anomaly']

def calculate_metrics(y_true, y_pred, method_name):
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    tn = ((y_true == 0) & (y_pred == 0)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / len(y_true)
    
    print(f"{method_name}:")
    print(f"  Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f} | Accuracy: {accuracy:.3f}")
    return precision, recall, f1, accuracy

dbscan_metrics = calculate_metrics(y_true, y_pred_dbscan, "DBSCAN")
iso_metrics = calculate_metrics(y_true, y_pred_iso, "Isolation Forest")

print("\nSTEP 10: INTERPRETING ANOMALY CHARACTERISTICS")
print("-" * 60)

anomaly_engines = df_aggregated[df_aggregated['iso_forest_anomaly'] == 1]
normal_engines = df_aggregated[df_aggregated['iso_forest_anomaly'] == 0]

print(f"Anomaly Analysis:")
print(f"  Anomalous engines: {len(anomaly_engines)}")
print(f"  Normal engines: {len(normal_engines)}")

print(f"\nSensor Characteristics Comparison:")
analysis_sensors = ['T30_mean', 'T50_mean', 'P30_mean', 'Nf_mean', 'Nc_mean']
for sensor in analysis_sensors:
    if sensor in df_aggregated.columns:
        anomaly_mean = anomaly_engines[sensor].mean()
        normal_mean = normal_engines[sensor].mean()
        difference = anomaly_mean - normal_mean
        percent_change = (difference / normal_mean * 100) if normal_mean != 0 else 0
        
        print(f"  {sensor}: Anomaly={anomaly_mean:.1f}, Normal={normal_mean:.1f}, "
              f"Diff={difference:.1f} ({percent_change:+.1f}%)")


print("\n" + "=" * 80)
print("ANALYSIS SUMMARY AND CONCLUSIONS")
print("=" * 80)


try:
    df_aggregated.to_csv('turbofan_anomaly_results.csv', index=False)
    
    performance_df = pd.DataFrame({
        'Method': ['DBSCAN', 'Isolation Forest'],
        'Precision': [dbscan_metrics[0], iso_metrics[0]],
        'Recall': [dbscan_metrics[1], iso_metrics[1]],
        'F1_Score': [dbscan_metrics[2], iso_metrics[2]],
        'Accuracy': [dbscan_metrics[3], iso_metrics[3]]
    })
    performance_df.to_csv('anomaly_detection_performance.csv', index=False)
    
    print(f"\nFiles saved:")
    print(f"  - turbofan_anomaly_results.csv")
    print(f"  - anomaly_detection_performance.csv")
except Exception as e:
    print(f"Error saving files: {e}")

print(f"\n" + "=" * 80)
print("END OF ANALYSIS")
print("=" * 80)

NASA TURBOFAN ENGINE ANOMALY DETECTION ANALYSIS
Using DBSCAN and Isolation Forest for Engine Failure Prediction

STEP 1: UNDERSTANDING SENSOR READINGS AND UNIT IDs
------------------------------------------------------------
Creating synthetic NASA Turbofan dataset...
(Replace this section with: df = pd.read_csv('NASA_Turbofan_Dataset.csv'))
✓ Dataset: 13134 observations from 100 engines
✓ Sensors: 17 sensor measurements
✓ Anomaly rate: 23.6%

STEP 2: CLEANING AND PREPROCESSING TIME-SERIES DATA
------------------------------------------------------------
Data Quality Check:
Missing values: 0
Duplicate rows: 0
Total outlier data points detected: 1429
✓ Data preprocessing completed

STEP 3: AGGREGATING DATA BY CYCLE AND STATISTICAL SUMMARIES
------------------------------------------------------------
✓ Aggregated dataset: 100 engines with 52 features

STEP 4: DIMENSIONALITY REDUCTION USING PCA
------------------------------------------------------------
Original features: 50
PCA compone